*斜體文字*

In [ ]:
!pip install -q streamlit pyngrok


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import itertools
import random

# =====================
# Session 初始化
# =====================
if "data_init" not in st.session_state:
    risk = ["低風險：劑量稍高", "高風險：嚴重交互作用"]
    md = ["高壓：醫師強勢要求", "低壓：醫師表示自負責任"]
    pt = ["負向：患者咆哮", "正向：患者焦慮"]
    peer = ["孤立：建議別惹麻煩", "支持：支持你的判斷"]

    all_combs = list(itertools.product(risk, md, pt, peer))
    st.session_state.vignettes = random.sample(all_combs, 3)

    st.session_state.step = -1
    st.session_state.answers = []
    st.session_state.data_init = True

st.title("💊 藥學生專業認同調查")

# =====================
# Step -1：IRB 說明 + 同意書
# =====================
if st.session_state.step == -1:
    with st.form("irb_form"):
        st.subheader("研究說明與同意書")

        st.markdown("""
**研究目的**
本研究旨在了解藥學生在臨床壓力情境下的專業判斷與專業認同。

**研究方式**
您將閱讀數個模擬臨床情境，並評估您堅持專業判斷的可能性。

**風險與權益**
- 本研究不涉及任何醫療介入
- 問卷完全匿名
- 可隨時中止填寫

**資料使用**
僅用於學術研究與教育用途
        """)

        consent = st.checkbox("我已閱讀並同意參與本研究")
        submit = st.form_submit_button("我同意，開始填寫")

        if submit:
            if not consent:
                st.error("⚠️ 請先勾選同意書")
            else:
                st.session_state.step = 0
                st.rerun()

# =====================
# Step 0：基本資料
# =====================
elif st.session_state.step == 0:
    with st.form("info_form"):
        st.subheader("步驟 1：基本資料")

        year = st.selectbox(
            "您的年級",
            ["請選擇", "P1", "P2", "P3", "P4", "P5", "P6"]
        )

        submit = st.form_submit_button("進入測驗")

        if submit:
            if year == "請選擇":
                st.error("⚠️ 請選擇年級")
            else:
                st.session_state.user_year = year
                st.session_state.step = 1
                st.rerun()

# =====================
# Step 1–3：情境題
# =====================
elif 1 <= st.session_state.step <= 3:
    idx = st.session_state.step - 1
    v = st.session_state.vignettes[idx]

    st.subheader(f"情境 {st.session_state.step} / 3")

    with st.form(f"vignette_form_{st.session_state.step}"):

        st.info(f"""
**用藥風險**：{v[0]}
**醫師態度**：{v[1]}
**患者反應**：{v[2]}
**同儕氛圍**：{v[3]}
""")

        score = st.slider(
            "你會堅持專業判斷的可能性？",
            1, 7, 4
        )

        submit = st.form_submit_button("下一題")

        if submit:
            st.session_state.answers.append({
                "年級": st.session_state.user_year,
                "情境編號": st.session_state.step,
                "得分": score
            })
            st.session_state.step += 1
            st.rerun()

# =====================
# Step 4：完成頁
# =====================
else:
    st.success("✅ 問卷完成，感謝您的參與！")

    df = pd.DataFrame(st.session_state.answers)
    st.dataframe(df, use_container_width=True)

    st.download_button(
        "📥 下載結果（CSV）",
        df.to_csv(index=False, encoding="utf-8-sig"),
        file_name="pharmacy_professional_identity.csv",
        mime="text/csv"
    )

    if st.button("🔄 重新填寫"):
        st.session_state.clear()
        st.rerun()


Overwriting app.py


In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
!./cloudflared tunnel --url http://localhost:8501

2026-01-30T06:18:16Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-01-30T06:18:16Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-01-30T06:18:18Z INF +--------------------------------------------------------------------------------------------+
2026-01-30T06:18:18Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-01-30T06:18:18Z INF |  https://innocent-behavioral-application-handling.tryc